# Dataset B 3-Class Baselines with Strict LORO-CV

This notebook benchmarks the low-speed Dataset B under a strict 3-class protocol using leave-one-run-out cross-validation (LORO-CV).

Protocol enforced here:
- Input: data/processed/dataset_B_pruned.csv
- Apply run-3 split if needed (partA/partB)
- Keep labels only: dry_dirt_track, grass, smooth_terrain
- Drop run: log_20260223_142511.490
- Final run list must be exactly 4 runs
- KNOWN_DEGENERATE mapping is fail-fast checked per fold
- LabelEncoder fit on train labels only (per fold)
- StandardScaler is inside each model pipeline

This notebook is benchmark-only and intentionally does not retrain a final deployment model.

In [1]:
from __future__ import annotations

import json
import warnings
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError("xgboost is required for this notebook.") from exc

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "src").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RANDOM_STATE = 42
N_TOP_MI = 25
ENABLE_FOLD_LOCAL_TOP_MI = True
MI_NEIGHBORS = 5
RUN_IMBALANCE_COMPARISON = True
N_PERMUTATION_REPEATS = 8
CLASS_SAMPLE_WARN_THRESHOLD = 30

DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_B_pruned.csv"
TOP_MI_PATH = PROJECT_ROOT / "data" / "processed" / "top_mi_features.json"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports" / "models"

for out_dir in [RESULTS_DIR, REPORTS_DIR]:
    out_dir.mkdir(parents=True, exist_ok=True)

LABEL_COL = "label"
RUN_COL = "run_id"
ID_COLS = [
    "window_id",
    "run_id",
    "segment_id",
    "label",
    "t_start",
    "t_end",
    "n_samples",
]
LABEL_ORDER = ["dry_dirt_track", "grass", "smooth_terrain"]

RUN3_ID = "log_20260309_141435.414"
EXCLUDED_RUN = "log_20260223_142511.490"
FINAL_RUNS = [
    "log_20260226_102148.990",
    "log_20260309_141435.414_partA",
    "log_20260309_141435.414_partB",
    "log_20260326_120021.508",
]
KNOWN_DEGENERATE = {
    "log_20260226_102148.990": [],
    "log_20260309_141435.414_partA": [],
    "log_20260309_141435.414_partB": [],
    "log_20260326_120021.508": [],
}

print("Configuration loaded for Dataset B 3-class benchmark")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
from src.evaluation import (
    compute_fold_metrics,
    fmt_mean_std,
    make_base_models,
    row_normalize,
    summarize_group,
)


Configuration loaded for Dataset B 3-class benchmark
PROJECT_ROOT: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis


In [2]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing dataset: {DATASET_PATH}")
if not TOP_MI_PATH.exists():
    raise FileNotFoundError(f"Missing MI feature list: {TOP_MI_PATH}")

df = pd.read_csv(DATASET_PATH)

if RUN3_ID in set(df[RUN_COL].astype(str)):
    run3 = df[df[RUN_COL] == RUN3_ID].sort_values("t_start")
    midpoint = run3["t_start"].median()
    df.loc[(df[RUN_COL] == RUN3_ID) & (df["t_start"] <= midpoint), RUN_COL] = f"{RUN3_ID}_partA"
    df.loc[(df[RUN_COL] == RUN3_ID) & (df["t_start"] > midpoint), RUN_COL] = f"{RUN3_ID}_partB"

df = df[df[LABEL_COL].isin(LABEL_ORDER)].copy()
df = df[df[RUN_COL].astype(str) != EXCLUDED_RUN].copy()

with TOP_MI_PATH.open("r", encoding="utf-8") as f:
    top_mi_raw = json.load(f)

if isinstance(top_mi_raw, dict):
    if "top_features" in top_mi_raw and isinstance(top_mi_raw["top_features"], list):
        top_mi_features = list(top_mi_raw["top_features"])
    else:
        top_mi_features = list(top_mi_raw.keys())
elif isinstance(top_mi_raw, list):
    top_mi_features = list(top_mi_raw)
else:
    raise ValueError("Unsupported top_mi_features.json structure.")

missing_id_cols = [c for c in ID_COLS if c not in df.columns]
if missing_id_cols:
    raise KeyError(f"Missing ID columns in dataset: {missing_id_cols}")

labels_found = sorted(df[LABEL_COL].astype(str).unique().tolist())
if labels_found != sorted(LABEL_ORDER):
    raise ValueError(f"Label check failed. Expected {sorted(LABEL_ORDER)} got {labels_found}")

run_ids = sorted(df[RUN_COL].astype(str).unique().tolist())
if run_ids != sorted(FINAL_RUNS):
    raise ValueError(
        f"Final run list mismatch. Expected {sorted(FINAL_RUNS)} got {run_ids}."
    )

if set(run_ids) != set(KNOWN_DEGENERATE.keys()):
    raise ValueError("KNOWN_DEGENERATE keys must match final run IDs exactly.")

feature_cols_all = [c for c in df.columns if c not in ID_COLS]
feature_cols_top_mi_seed = [c for c in top_mi_features if c in feature_cols_all]
if not feature_cols_top_mi_seed:
    raise ValueError("No overlap between top_mi_features.json and dataset columns.")

speed_cols = [c for c in ["odom_mean_speed", "odom_speed_std", "odom_distance", "net_speed"] if c in feature_cols_all]
feature_cols_all_no_speed = [c for c in feature_cols_all if c not in set(speed_cols)]
if not feature_cols_all_no_speed:
    raise ValueError("all_no_speed feature set became empty.")

feature_sets = {
    "all": feature_cols_all,
    "all_no_speed": feature_cols_all_no_speed,
    "top_mi": feature_cols_top_mi_seed[:N_TOP_MI],
}

print(f"Dataset shape after filtering: {df.shape}")
print(f"Runs ({len(run_ids)}): {run_ids}")
print("Run x label distribution:")
print(df.groupby(RUN_COL)[LABEL_COL].value_counts().unstack(fill_value=0).to_string())
print(f"Feature counts - all: {len(feature_cols_all)}, all_no_speed: {len(feature_cols_all_no_speed)}, top_mi_seed: {len(feature_sets['top_mi'])}")

Dataset shape after filtering: (1259, 128)
Runs (4): ['log_20260226_102148.990', 'log_20260309_141435.414_partA', 'log_20260309_141435.414_partB', 'log_20260326_120021.508']
Run x label distribution:
label                          dry_dirt_track  grass  smooth_terrain
run_id                                                              
log_20260226_102148.990                     0     44              55
log_20260309_141435.414_partA             268     61             153
log_20260309_141435.414_partB             255    227               0
log_20260326_120021.508                     0    102              94
Feature counts - all: 121, all_no_speed: 118, top_mi_seed: 9


In [3]:
def build_model_variants(run_imbalance_comparison: bool = True) -> list[dict[str, object]]:
    variants: list[dict[str, object]] = []
    for model_name, pipeline in make_base_models(RANDOM_STATE).items():
        variants.append(
            {
                "variant_name": f"{model_name}_unweighted",
                "base_model": model_name,
                "imbalance": "unweighted",
                "pipeline": clone(pipeline),
                "requires_encoded_labels": model_name == "XGBoost",
                "needs_sample_weight": False,
            }
        )

        if run_imbalance_comparison:
            balanced_pipe = clone(pipeline)
            needs_sample_weight = False
            if model_name in {"LogisticRegression", "RandomForest"}:
                balanced_pipe.set_params(clf__class_weight="balanced")
            elif model_name == "XGBoost":
                needs_sample_weight = True

            variants.append(
                {
                    "variant_name": f"{model_name}_balanced",
                    "base_model": model_name,
                    "imbalance": "balanced",
                    "pipeline": balanced_pipe,
                    "requires_encoded_labels": model_name == "XGBoost",
                    "needs_sample_weight": needs_sample_weight,
                }
            )
    return variants


def fold_local_top_mi_features(
    train_df: pd.DataFrame,
    y_train: np.ndarray,
    pool_cols: list[str],
    n_top: int,
    n_neighbors: int,
    random_state: int,
    fallback_cols: list[str],
) -> list[str]:
    X_pool = np.nan_to_num(train_df[pool_cols].to_numpy(dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    if X_pool.shape[1] == 0:
        return list(fallback_cols)
    scores = mutual_info_classif(
        X_pool,
        y_train,
        n_neighbors=n_neighbors,
        random_state=random_state,
    )
    ranked_idx = np.argsort(scores)[::-1]
    selected = [pool_cols[i] for i in ranked_idx[: min(n_top, len(pool_cols))]]
    return selected if selected else list(fallback_cols)


Model variants: ['LogisticRegression_unweighted', 'LogisticRegression_balanced', 'RandomForest_unweighted', 'RandomForest_balanced', 'XGBoost_unweighted', 'XGBoost_balanced']


In [9]:
detail_rows: list[dict[str, object]] = []
per_class_rows: list[dict[str, object]] = []
confusion_records: list[dict[str, object]] = []
fold_top_mi_rows: list[dict[str, object]] = []
feature_importance_store: dict[tuple[str, str, str], list[pd.Series]] = {}
permutation_importance_store: dict[tuple[str, str, str], list[pd.Series]] = {}
lr_coef_store: dict[tuple[str, str], list[pd.DataFrame]] = {}

for feature_set_name, base_feature_cols in feature_sets.items():
    for test_run in run_ids:
        train_mask = df[RUN_COL].astype(str) != test_run
        test_mask = ~train_mask

        train_df = df.loc[train_mask].copy()
        test_df = df.loc[test_mask].copy()

        y_train = train_df[LABEL_COL].astype(str).to_numpy()
        y_test = test_df[LABEL_COL].astype(str).to_numpy()

        observed_missing_from_train = sorted(set(y_test) - set(y_train))
        expected_missing = sorted(KNOWN_DEGENERATE[test_run])
        if observed_missing_from_train != expected_missing:
            raise ValueError(
                f"KNOWN_DEGENERATE mismatch for {test_run}. Expected {expected_missing}, observed {observed_missing_from_train}."
            )

        selected_feature_cols = list(base_feature_cols)
        if feature_set_name == "top_mi" and ENABLE_FOLD_LOCAL_TOP_MI:
            selected_feature_cols = fold_local_top_mi_features(
                train_df=train_df,
                y_train=y_train,
                pool_cols=feature_cols_all,
                n_top=N_TOP_MI,
                n_neighbors=MI_NEIGHBORS,
                random_state=RANDOM_STATE,
                fallback_cols=feature_sets["top_mi"],
            )
            fold_top_mi_rows.append(
                {
                    "test_run": test_run,
                    "n_features": int(len(selected_feature_cols)),
                    "selected_features": "|".join(selected_feature_cols),
                }
            )

        X_train = train_df[selected_feature_cols]
        X_test = test_df[selected_feature_cols]

        le = LabelEncoder()
        y_train_encoded = le.fit_transform(y_train)
        encoder_map = {label: idx for idx, label in enumerate(le.classes_)}
        y_test_encoded = np.array([encoder_map.get(label, -1) for label in y_test], dtype=int)
        if np.any(y_test_encoded < 0):
            raise ValueError(f"Unexpected unseen test label in fold {test_run}.")

        for variant in model_variants:
            pipeline = clone(variant["pipeline"])
            base_model = str(variant["base_model"])
            imbalance = str(variant["imbalance"])
            variant_name = str(variant["variant_name"])

            fit_kwargs = {}
            if bool(variant["needs_sample_weight"]):
                fit_kwargs["clf__sample_weight"] = compute_sample_weight("balanced", y_train_encoded)

            if bool(variant["requires_encoded_labels"]):
                pipeline.fit(X_train, y_train_encoded, **fit_kwargs)
                y_pred_encoded = np.asarray(pipeline.predict(X_test), dtype=int)
                y_pred = le.inverse_transform(y_pred_encoded)
                y_perm_target = y_test_encoded
            else:
                pipeline.fit(X_train, y_train)
                y_pred = np.asarray(pipeline.predict(X_test), dtype=object)
                y_perm_target = y_test

            evaluable_classes = sorted(set(y_train).intersection(set(y_test)))
            fold_metrics = compute_fold_metrics(
                y_true=y_test,
                y_pred=y_pred,
                evaluable_classes=evaluable_classes,
                label_order=LABEL_ORDER,
            )

            detail_rows.append(
                {
                    "model": base_model,
                    "feature_set": feature_set_name,
                    "imbalance": imbalance,
                    "test_run": test_run,
                    "n_test": fold_metrics["n_test"],
                    "n_features": int(len(selected_feature_cols)),
                    "accuracy": fold_metrics["accuracy"],
                    "macro_f1": fold_metrics["macro_f1"],
                    "degenerate_classes": "None",
                }
            )

            confusion_records.append(
                {
                    "model": base_model,
                    "feature_set": feature_set_name,
                    "imbalance": imbalance,
                    "test_run": test_run,
                    "y_true": y_test.copy(),
                    "y_pred": y_pred.copy(),
                    "degenerate_classes": expected_missing,
                }
            )

            for cls_name in LABEL_ORDER:
                cls_metrics = fold_metrics["per_class"][cls_name]
                per_class_rows.append(
                    {
                        "model": base_model,
                        "feature_set": feature_set_name,
                        "imbalance": imbalance,
                        "test_run": test_run,
                        "class": cls_name,
                        "precision": cls_metrics["precision"],
                        "recall": cls_metrics["recall"],
                        "f1": cls_metrics["f1"],
                        "support": cls_metrics["support"],
                    }
                )

            try:
                perm_result = permutation_importance(
                    estimator=pipeline,
                    X=X_test,
                    y=y_perm_target,
                    scoring="f1_macro",
                    n_repeats=N_PERMUTATION_REPEATS,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )
                perm_series = pd.Series(perm_result.importances_mean, index=selected_feature_cols, dtype=float)
                permutation_importance_store.setdefault((base_model, feature_set_name, imbalance), []).append(perm_series)
            except Exception as exc:
                warnings.warn(
                    f"Permutation importance failed for {variant_name} on {test_run} ({feature_set_name}): {exc}"
                )

            clf = pipeline.named_steps["clf"]
            if base_model in {"RandomForest", "XGBoost"} and hasattr(clf, "feature_importances_"):
                imp = pd.Series(clf.feature_importances_, index=selected_feature_cols, dtype=float)
                feature_importance_store.setdefault((base_model, feature_set_name, imbalance), []).append(imp)
            elif base_model == "LogisticRegression":
                coef_df = pd.DataFrame(clf.coef_, index=clf.classes_, columns=selected_feature_cols)
                lr_coef_store.setdefault((feature_set_name, imbalance), []).append(coef_df)

            print(
                f"Fold={test_run} | {variant_name} | feature_set={feature_set_name} | n_feat={len(selected_feature_cols)} | "
                f"acc={fold_metrics['accuracy']:.3f} | macro_f1={fold_metrics['macro_f1']:.3f}"
            )

if fold_top_mi_rows:
    fold_top_mi_df = pd.DataFrame(fold_top_mi_rows)
    fold_top_mi_out = RESULTS_DIR / "dataset_B_3class_fold_local_top_mi_features.csv"
    fold_top_mi_df.to_csv(fold_top_mi_out, index=False)
    print(f"Saved: {fold_top_mi_out}")

print("LORO evaluation complete.")

Fold=log_20260226_102148.990 | LogisticRegression_unweighted | feature_set=all | n_feat=121 | acc=0.970 | macro_f1=0.984
Fold=log_20260226_102148.990 | LogisticRegression_balanced | feature_set=all | n_feat=121 | acc=0.980 | macro_f1=0.988
Fold=log_20260226_102148.990 | RandomForest_unweighted | feature_set=all | n_feat=121 | acc=0.980 | macro_f1=0.991
Fold=log_20260226_102148.990 | RandomForest_balanced | feature_set=all | n_feat=121 | acc=0.970 | macro_f1=0.986
Fold=log_20260226_102148.990 | XGBoost_unweighted | feature_set=all | n_feat=121 | acc=0.990 | macro_f1=0.995
Fold=log_20260226_102148.990 | XGBoost_balanced | feature_set=all | n_feat=121 | acc=0.980 | macro_f1=0.991
Fold=log_20260309_141435.414_partA | LogisticRegression_unweighted | feature_set=all | n_feat=121 | acc=0.940 | macro_f1=0.935
Fold=log_20260309_141435.414_partA | LogisticRegression_balanced | feature_set=all | n_feat=121 | acc=0.944 | macro_f1=0.940
Fold=log_20260309_141435.414_partA | RandomForest_unweighted |

In [13]:
a_summary_path = RESULTS_DIR / "baseline_metrics_summary.csv"
if not a_summary_path.exists():
    raise FileNotFoundError(f"Missing Dataset A reference summary: {a_summary_path}")

a_summary = pd.read_csv(a_summary_path)
required_cols = {"model", "feature_set", "imbalance", "macro_f1_mean"}
missing_cols = required_cols - set(a_summary.columns)
if missing_cols:
    raise ValueError(f"Dataset A summary missing required columns: {sorted(missing_cols)}")

a_all = a_summary[a_summary["feature_set"] == "all"][["model", "imbalance", "macro_f1_mean"]].copy()
a_all = a_all.rename(columns={"macro_f1_mean": "macro_f1_mean_A_all"})

b_all = summary_df[summary_df["feature_set"] == "all"][["model", "imbalance", "macro_f1_mean", "macro_f1_std", "worst_fold_macro_f1", "robustness_score"]].copy()
b_all = b_all.rename(columns={"macro_f1_mean": "macro_f1_mean_B_3class"})

comparison = b_all.merge(a_all, on=["model", "imbalance"], how="left")
comparison["delta_macro_f1_B_minus_A"] = comparison["macro_f1_mean_B_3class"] - comparison["macro_f1_mean_A_all"]

comparison_out = RESULTS_DIR / "dataset_B_3class_vs_A_all_contextual_comparison.csv"
comparison.to_csv(comparison_out, index=False)

print("Dataset B 3-class summary (feature_set=all):")
print(
    b_all.sort_values(["model", "imbalance"]).to_string(index=False)
)
print("\nContextual comparison vs Dataset A (feature_set=all):")
print(
    comparison.sort_values(["model", "imbalance"]).to_string(index=False)
)
print("\nCaveat: Delta macro F1 is contextual only because Dataset B here is 3-class and low-speed filtered, while Dataset A baseline is a different label/speed regime.")
print("This notebook intentionally does not retrain a final model; use results for benchmark interpretation only.")
print(f"Saved: {comparison_out}")

Dataset B 3-class summary (feature_set=all):
             model  imbalance  macro_f1_mean_B_3class  macro_f1_std  worst_fold_macro_f1  robustness_score
LogisticRegression   balanced                0.970239      0.020614             0.939835          0.959932
LogisticRegression unweighted                0.968397      0.021419             0.934943          0.957687
      RandomForest   balanced                0.977830      0.019789             0.943951          0.967936
      RandomForest unweighted                0.979213      0.021443             0.942162          0.968492
           XGBoost   balanced                0.978829      0.013694             0.955689          0.971981
           XGBoost unweighted                0.980970      0.014338             0.957484          0.973801

Contextual comparison vs Dataset A (feature_set=all):
             model  imbalance  macro_f1_mean_B_3class  macro_f1_std  worst_fold_macro_f1  robustness_score  macro_f1_mean_A_all  delta_macro_f1_B_minus